# Lesson 2: Aggregations, GROUP BY, HAVING, CASE WHEN

**Week 2 · Data Engineering Course**

---

**Prerequisites:** Lab 2 complete, Lesson 1 (JOINs) read.

**What you will learn:**
- Aggregate functions: COUNT, SUM, AVG, MIN, MAX
- GROUP BY — summarise data by category
- HAVING — filter groups after aggregation
- CASE WHEN — conditional logic inside queries

**All SQL runs in pgAdmin → Tools → Query Tool → select `de_course`**

---

## 2.1 Aggregate Functions

Aggregate functions collapse many rows into a single summary value.

| Function | Returns |
|----------|---------|
| `COUNT(*)` | Number of rows |
| `COUNT(column)` | Number of non-NULL values in that column |
| `SUM(column)` | Total of all values |
| `AVG(column)` | Mean value |
| `MIN(column)` | Smallest value |
| `MAX(column)` | Largest value |

**Examples — run each one separately in pgAdmin:**

```sql
-- Total number of orders
SELECT COUNT(*) AS total_orders FROM orders;

-- Highest product price
SELECT MAX(price) AS highest_price FROM products;

-- Average order item unit price
SELECT ROUND(AVG(unit_price), 2) AS avg_unit_price FROM order_items;

-- Total revenue across all order_items
SELECT SUM(quantity * unit_price) AS total_revenue FROM order_items;
```

> `ROUND(value, 2)` rounds to 2 decimal places.

---

## 2.2 GROUP BY

GROUP BY splits rows into groups before aggregating. You get one result row per group.

**Syntax:**

```sql
SELECT group_column, aggregate_function()
FROM table
GROUP BY group_column;
```

**Example — order count per status:**

```sql
SELECT
    status,
    COUNT(*) AS order_count
FROM orders
GROUP BY status
ORDER BY order_count DESC;
```

Expected result:

| status | order_count |
|--------|-------------|
| completed | 38 |
| pending | 6 |
| cancelled | 4 |
| refunded | 2 |

**Example — revenue per product category:**

```sql
SELECT
    p.category,
    SUM(oi.quantity * oi.unit_price) AS total_revenue
FROM order_items oi
INNER JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
```

---

### The SELECT rule with GROUP BY

Every column in SELECT must be either:
1. In the GROUP BY clause, **or**
2. Inside an aggregate function

```sql
-- Wrong: first_name is not in GROUP BY and not aggregated
SELECT first_name, status, COUNT(*)
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
GROUP BY status;

-- Right: first_name added to GROUP BY
SELECT c.first_name, o.status, COUNT(*) AS order_count
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.first_name, o.status
ORDER BY c.first_name;
```

This is one of the most common SQL errors for beginners. PostgreSQL will tell you exactly which column is the problem.

---

### GROUP BY with multiple columns

You can group by more than one column. Each unique combination forms its own group.

**Example — order count per salesperson per status:**

```sql
SELECT
    s.first_name || ' ' || s.last_name AS salesperson,
    o.status,
    COUNT(*) AS order_count
FROM orders o
INNER JOIN salespeople s ON o.salesperson_id = s.salesperson_id
GROUP BY s.first_name, s.last_name, o.status
ORDER BY salesperson, o.status;
```

**Example — total revenue per salesperson:**

```sql
SELECT
    s.first_name || ' ' || s.last_name AS salesperson,
    s.region,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(oi.quantity * oi.unit_price) AS total_revenue
FROM salespeople s
LEFT JOIN orders o ON s.salesperson_id = o.salesperson_id
LEFT JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY s.salesperson_id, s.first_name, s.last_name, s.region
ORDER BY total_revenue DESC NULLS LAST;
```

> `NULLS LAST` puts NULL revenue at the bottom rather than the top.

---

## 2.3 HAVING

WHERE filters **rows before** grouping. HAVING filters **groups after** aggregation.

```sql
SELECT group_column, COUNT(*)
FROM table
WHERE row_condition          -- runs first, removes individual rows
GROUP BY group_column
HAVING group_condition       -- runs after grouping, removes entire groups
ORDER BY ...;
```

**Example — customers who placed more than 2 orders:**

```sql
SELECT
    c.first_name || ' ' || c.last_name AS customer,
    COUNT(o.order_id) AS order_count
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING COUNT(o.order_id) > 2
ORDER BY order_count DESC;
```

**Example — product categories with total revenue above $5,000:**

```sql
SELECT
    p.category,
    SUM(oi.quantity * oi.unit_price) AS category_revenue
FROM order_items oi
INNER JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
HAVING SUM(oi.quantity * oi.unit_price) > 5000
ORDER BY category_revenue DESC;
```

> A common mistake: using WHERE to filter on an aggregated value. `WHERE COUNT(*) > 2` is invalid. Use HAVING.

---

## 2.4 CASE WHEN

CASE WHEN adds conditional logic inside a query — like an IF/ELSE in programming, or an IF formula in Excel.

**Syntax:**

```sql
CASE
    WHEN condition_1 THEN result_1
    WHEN condition_2 THEN result_2
    ELSE default_result
END AS column_alias
```

**Example — categorise products by price tier:**

```sql
SELECT
    product_name,
    price,
    CASE
        WHEN price < 50  THEN 'Budget'
        WHEN price < 200 THEN 'Mid-range'
        ELSE                  'Premium'
    END AS price_tier
FROM products
ORDER BY price;
```

**Example — label each order by value size:**

```sql
SELECT
    o.order_id,
    SUM(oi.quantity * oi.unit_price) AS order_total,
    CASE
        WHEN SUM(oi.quantity * oi.unit_price) < 100   THEN 'Small'
        WHEN SUM(oi.quantity * oi.unit_price) < 500   THEN 'Medium'
        ELSE                                               'Large'
    END AS order_size
FROM orders o
INNER JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY o.order_id
ORDER BY order_total DESC;
```

---

### CASE WHEN inside aggregate functions

Combining CASE WHEN with SUM or COUNT lets you pivot data — counting or summing only rows that meet a condition.

**Example — completed vs cancelled orders per salesperson in one row:**

```sql
SELECT
    s.first_name || ' ' || s.last_name AS salesperson,
    COUNT(*) AS total_orders,
    SUM(CASE WHEN o.status = 'completed'  THEN 1 ELSE 0 END) AS completed,
    SUM(CASE WHEN o.status = 'cancelled'  THEN 1 ELSE 0 END) AS cancelled,
    SUM(CASE WHEN o.status = 'pending'    THEN 1 ELSE 0 END) AS pending
FROM orders o
INNER JOIN salespeople s ON o.salesperson_id = s.salesperson_id
GROUP BY s.salesperson_id, s.first_name, s.last_name
ORDER BY total_orders DESC;
```

This pattern — `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` — is called **conditional aggregation**. It is one of the most useful patterns in analytical SQL.

**Example — revenue from completed orders only:**

```sql
SELECT
    SUM(CASE WHEN o.status = 'completed'
             THEN oi.quantity * oi.unit_price
             ELSE 0 END) AS completed_revenue,
    SUM(oi.quantity * oi.unit_price) AS total_revenue
FROM order_items oi
INNER JOIN orders o ON oi.order_id = o.order_id;
```

---

## 2.5 The Order SQL Executes

SQL clauses execute in a specific order — different from the order you write them:

```
1. FROM / JOIN     -- which tables, how joined
2. WHERE           -- filter rows
3. GROUP BY        -- form groups
4. HAVING          -- filter groups
5. SELECT          -- compute output columns
6. ORDER BY        -- sort
7. LIMIT           -- truncate
```

This explains several rules:
- You cannot use a SELECT alias in WHERE (WHERE runs before SELECT)
- You cannot use a SELECT alias in HAVING for the same reason
- You CAN use a SELECT alias in ORDER BY (ORDER BY runs after SELECT)

```sql
-- Wrong: alias 'revenue' used in HAVING -- runs before SELECT
SELECT p.category, SUM(oi.quantity * oi.unit_price) AS revenue
FROM order_items oi INNER JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
HAVING revenue > 5000;  -- error

-- Right: repeat the expression in HAVING
HAVING SUM(oi.quantity * oi.unit_price) > 5000;
```

---

## 2.6 Key Takeaways

1. **Aggregate functions** (COUNT, SUM, AVG, MIN, MAX) collapse many rows into one summary value.
2. **GROUP BY** creates one output row per unique combination of the grouped columns.
3. Every column in SELECT must be either in GROUP BY or inside an aggregate — no exceptions.
4. **WHERE** filters rows before grouping; **HAVING** filters groups after aggregation.
5. **CASE WHEN** adds IF/ELSE logic inside a query. It can appear inside SELECT, ORDER BY, and even inside aggregate functions.
6. **Conditional aggregation** (`SUM(CASE WHEN ... THEN 1 ELSE 0 END)`) pivots row-level data into summary columns.
7. SQL executes in the order: FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT. This determines what you can reference where.